# Exploratory data analysis on raw TPM data

RNA-seq data was collected and ...

## Data extraction

In [5]:
# All functions for data conversion and extraction from specified path
import pandas as pd
import numpy as np
import os
import functools

from functools import reduce

def read_fcnts_as_df(folder_path):
    """
    Extracts all feature count csvs from a specified folder and stores them as a list of dataframes

    Args:
        folder_path : File path containing Fcnts csv output from pipeline

    Returns: 
        fcnt_df_list : List of Fcnts dataframes
    """
    
    # Filter out the summary files and keep Fcnts from no-drug control (NDC) comparisons
    all_files = os.listdir(folder_path)
    files = [
        f for f in all_files
        if f.endswith(".csv")
        and ".summary" not in f
        and "NDC0hr" in f
            ]

    if len(files) == 0: 
        raise FileNotFoundError(f"No matching feature count files found in {folder_path}")

    # Convert each csv (which is stored as tab-delimited) to dataframe
    files = sorted(files)
    filenames = ["".join([folder_path, "/" , f]) for f in files]
    fcnt_df_list = [pd.read_table(f, sep = "\t", header = 0, skiprows = 1) for f in filenames]

    return fcnt_df_list

def sample_name_strip(name):
    """
    Convert a sample file name into an easy to read sample name
    Args:
        name : (ex: "/ExpOut/260107_AV242502_RNASeq_miniHT_SpnT4WT_CEF_CIP/Out/Rep/Bams/T4-wt12CEF12CIP1hr-a.bam")
        
    Returns:
        new_name : (ex: 12CEF12CIP1hr-a)
    """
    # Find index of the last / and remove entire prefix (OG file path)
    samplename_start_idx = name.strip().rfind("/") + 1
    new_name = name[samplename_start_idx:]

    # Find index of . (.bam is at end of sample name) and remove filetag
    filetag_start_idx = new_name.rfind(".")
    new_name = new_name[:filetag_start_idx]

    # Remove "T4-wt"
    new_name = new_name.replace("T4-wt", "")

    return new_name

def fcnts_to_tpms(fcnt_df_list):
    """
    Converts a list of RNA-seq feature count dataframes to a list of TPM dataframes

    Args:
        fcnt_df_list : List of feature count dataframes

    Returns:
        tpm_df_list : List of TPM dataframes

    """
    tpm_df_list = []

    for df in fcnt_df_list:

        # Move gene names to index and keep only length, Fcnts
        df = df.set_index("Geneid")
        df = df.loc[:, [col for col in list(df.columns) if col == "Length" or col.startswith("/")]]
        df = df.apply(lambda column: [int(entry) for entry in column])

        # Convert gene length from b -> kb
        df["Length"] = df["Length"].apply(lambda column: column / 1000)

        # Select Fcnts columns by excluding Length column
        fcnts_cols = [col for col in list(df.columns) if col != "Length"]
        
        # Fcnts / gene length = (Counts per kb)
        df[fcnts_cols] = df[fcnts_cols].apply(lambda column: column / df["Length"])

        # (Counts per kb) * 10^6 / (Total counts/kb) = TPM
        df[fcnts_cols] = df[fcnts_cols].apply(lambda column: column * 10**6 / sum(column))

        # Remove length column 
        df.drop(columns = "Length", inplace = True)
        
        # Apply stripper to each column names
        df = df.rename(columns = lambda column: sample_name_strip(column))
        tpm_df_list.append(df)        

    return tpm_df_list

def bind_tpm_data(tpm_df_list):
    """
    Function to take a list of TPM dataframes, then bind all into 1 dataframe
    Args: 
        tpm_df_list : list of TPM dataframes

    Returns:
        all_tpms [N,G] : dataframe with all TPM values (N samples on row, G genes on column)
    """
    tpm_df_list_uniq = []

    # Column names of 1st DF (all have last 3 cols)
    colnames = list(tpm_df_list[0].columns)

    # Select redundant NDC0hr columns and make new df with just those
    ndc0hr_cols = [col for col in colnames if "NDC0hr" in col]
    ndc0hr_df = tpm_df_list[0][ndc0hr_cols]
    tpm_df_list_uniq.append(ndc0hr_df)

    # Remove NDC0hr columns from all dfs
    for df in tpm_df_list:
        columns = list(df.columns)
        relevant_idx = [col for col in columns if "NDC0hr" not in col]
        stripped_df = df[relevant_idx]
        tpm_df_list_uniq.append(stripped_df)

    all_tpms = reduce(lambda df1, df2 :
                      pd.merge(df1, df2, 
                               left_index = True, 
                               right_index = True, 
                               how = "outer"),
                      tpm_df_list_uniq)

    # Tranpose to get genes on columns
    all_tpms = all_tpms.T

    return all_tpms

def read_cfus(folder_path):
    """
    Function to extract CFUs into a dataframe with conditions on rows, 1 CFU column
    Args:
        folder_path : path to folder containing CFUs (same format as /all_cfus)

    Returns:
        all_cfus [N,1] : df with condition names as index, 1 column of CFUs (N = # samples)
    """

    # Get files
    files = os.listdir(folder_path)
    
    # Select CSV files
    cfu_files = [csv for csv in files if ".csv" in csv]
    cfu_files = sorted(cfu_files)
    cfu_files = ["".join([folder_path, "/", csv]) for csv in files]
    
    # Load each file as a dataframe
    cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files] 

    for i, df in enumerate(cfu_dfs):

        # Melt to get all triplicates stacked
        df = df.melt(id_vars = "Triplicates", var_name = "Condition", value_name = "CFU")

        # Define labels by combining condition + "-" + triplicate label
        df["Labels"] = df["Condition"].str.strip() + "-" + df["Triplicates"].str.strip()

        # Drop unncessary columns
        df = df.drop(columns = ["Condition", "Triplicates"])

        # Move labels to index
        df = df.set_index("Labels")
        cfu_dfs[i] = df
    
    # Concat
    all_cfus = pd.concat(cfu_dfs)

    return all_cfus

# Function to bind TPMs
def bind_all_data(tpm_df, cfu_df):
    """
    Function bind TPM and cfu dfs
    Args:
        tpm_df [N,G] : Dataframe of TPMs, N = # samples, G = # genes, labels on index
        cfu_df [N,1] : Dataframe of CFUs, labels on index

    Returns:
        data_df : [N, G+1] : Dataframe of all TPMs and CFUs as last column
    """
    # Right join so that CFUs exist
    data_df = pd.merge(tpm_df, cfu_df, left_index = True, right_index = True, how = "right")

    return data_df

def data_extract(fcnts_path, cfu_path):
    """
    Function to run entire data extraction pipeline

    Args:
        fcnts_path : Path to folder containing Fcnts output of lab pipeline
        cfu_path   : Path to folder containing CFU csv outputs

    Returns:
        all_data : [N, G+1] : Dataframe of all TPMs and CFUs as last column 
    """
    stored_fcnts = read_fcnts_as_df(fcnts_path)
    stored_tpms  = fcnts_to_tpms(stored_fcnts)
    tpm_df       = bind_tpm_data(stored_tpms)
    cfu_df       = read_cfus(cfu_path)
    all_data     = bind_all_data(tpm_df, cfu_df)
    
    return all_data

We'll apply our data loading pipeline to the our Fcnts and CFU filepaths.

In [6]:
# Paths to Fcnts and CFU data
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"
cfu_path   = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"

# Run data pipeline
stored_fcnts = read_fcnts_as_df(fcnts_path)
stored_tpms  = fcnts_to_tpms(stored_fcnts)
tpm_df       = bind_tpm_data(stored_tpms)
cfu_df       = read_cfus(cfu_path)
data_df      = bind_all_data(tpm_df, cfu_df)
    
# Data dimensions
print(f"Number of samples          : {data_df.shape[0]}")
print(f"Number of features (genes) : {data_df.shape[1] - 1}")
data_df.head(n = 5)

Number of samples          : 480
Number of features (genes) : 2191


,F01-SP0085-SP0086,F02-SP0103-SP0104,F03-SP0115-SP0116,F04-SP0116-SP0117,F05-SP0117-SP0118,F06-SP0129-SP0130,F07-SP0239-SP0240,F08-SP0256-SP0257,F09-SP0257-SP0258,F10-SP0311-SP0312,...,SP_2231,SP_2233,SP_2234,SP_2235,SP_2236,SP_2237,SP_2238,SP_2239,SP_2240,CFU
Labels,,,,,,,,,,,,,,,,,,,,,
NDC1hr-a,17.873518,2466.545463,494.725485,32.828910,1429.881428,16.932806,2276.344255,363.428196,263.813123,1879.541509,...,64.521089,6.283659,29.032123,13.244918,22.321678,2.553360,18.096937,1232.456175,1609.040485,580000000
NDC1hr-b,8.599118,2848.027779,461.431900,25.270876,1792.237161,32.586130,1828.204870,366.895688,383.864614,2118.098463,...,65.478593,4.030836,25.084719,17.266754,20.077578,0.000000,28.377088,1234.082514,1664.082222,1000000000
NDC1hr-c,6.976092,2288.158311,559.140388,41.002339,1462.776430,8.811906,2061.237871,288.345153,411.868496,1868.124115,...,64.530904,5.232069,26.825195,17.342955,21.212371,11.959016,28.253174,1369.580457,1827.791359,800000000
14CEF1hr-a,2.366695,2718.858963,323.120819,20.865554,628.793434,5.979018,4015.700532,485.961328,201.074389,893.863250,...,35.642257,3.328165,19.456585,12.898958,10.794698,16.228764,21.300253,1958.037472,1537.434864,340000000
14CEF1hr-b,14.497379,2493.549214,295.418292,15.976704,860.686510,0.000000,1878.367974,604.057465,151.352638,819.483432,...,42.827745,2.038694,15.378435,12.129281,14.956572,0.000000,24.464327,1769.710527,1570.874119,700000000


Separate data into sample x gene matrix and CFU values.

In [24]:
# Columns of data matrix
cols = data_df.columns
rna_data = data_df[[col for col in cols if col != "CFU"]]
cfu_data = data_df[[col for col in cols if col == "CFU"]]

## Data annotation

Define function to help convert condition labels to drug and timepoint.

In [17]:
# Define functions to convert condition labels to drug and to 
def find_first_alpha(str):
    """
    Function to find the index of the the first letter in a string
    (will be used for drug name parser function)

    Args:
        str : string

    Returns:
        first_alpha : idx of first letter
    """
    for i in range(len(str)):
        if str[i].isalpha():
            first_alpha = i
            return first_alpha
            break 

def condition_to_drug(condition_label):
    """
    Function to convert a condition label to a drug name

    Args:
        condition_label : Condition label, ex: "12CIP1hr-a"

    Returns:
        drug_name : Drug name, ex: "CIP"
    """
    # Single drug case
    if len(condition_label) < 11:

        # If NDC sample
        if "NDC" in condition_label:
            return "NDC"
        
        else:

            # Find drug name using letter search
            first_alpha_idx = find_first_alpha(condition_label)
            drug_name = condition_label[first_alpha_idx:first_alpha_idx + 3]
            return drug_name
    
    # Multiple drug case
    else:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        drug1 = condition_label[first_idx:first_idx + 3]

        # Find first letter in remaining string and extract drug name
        substr     = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        drug2      = substr[second_idx:second_idx + 3]

        # Merge drug names
        drug_name = drug1 + "+" + drug2

        return drug_name
    
def condition_to_time(condition_label):
    """
    Function to convert condition label to timepoint

    Args:
        condition_label : Condition label, ex: "12CIP1hr-a"
    
    Returns:
        timepoint : ex: "1hr"
    """
    time_idx = condition_label.find("hr")
    timepoint = condition_label[time_idx-1:time_idx + 2]

    return timepoint

def condition_to_dose(condition_label):
    """
    Function to convert condition label to dose

    Args: 
        
    Returns:

    """

Apply and add labels to data for visualization.

In [30]:
# Add columns for each metadata aspect
drug_labels = [condition_to_drug(cond) for cond in rna_data.index]
time_labels = [condition_to_time(cond) for cond in rna_data.index]

## PCA

In [ ]:
# Create scaler and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize scaler for data centering
scaler = StandardScaler(with_std = False)

# Make new centered df
transformed_data = rna_data.copy()
transformed_data = scaler.fit_transform(transformed_data)

# PCA object
pca = PCA(n_components = 2)

# Make PCA transformed df
transformed_data = pca.fit_transform(transformed_data)

# Find percent variance


transformed_df = pd.DataFrame({"PC1": transformed_data[:,0],
                               "PC2": transformed_data[:,1]
                               })
transformed_df["Drug"] = drug_labels
transformed_df["Timepoint"] = time_labels

# Plot  
sns.scatterplot(transformed_data, x = "0", y = "1", )

ValueError: Could not interpret value `0` for `x`. An entry with this name does not appear in `data`.

In [ ]:
sns.scatterplot()

,0,1
0,3765.207376,-18914.191045
1,1420.212335,-23111.902624
2,6877.352154,-16834.751446
3,-12294.813154,-26806.164225
4,-6394.988625,-20869.966724
...,...,...
475,-21505.399341,40233.231510
476,-22607.922734,35304.625596
477,-15481.815668,19508.234618
478,-10100.366421,21889.755280


## Heatmap

## Feature correlation

## CFU across timepoint?